In [8]:
import pandas as pd  
import numpy as np  
import os  
import re  
import glob  
import io
from datetime import datetime  
from dotenv import load_dotenv  
import msoffcrypto
import warnings
from dateutil.relativedelta import relativedelta
from msoffcrypto.exceptions import FileFormatError, InvalidKeyError

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

In [ ]:
# ---------- Settings ----------  
FILES = {  
    2025: 'L:/2025 Pilar Plant Files/Monthly Expense Reports/12. December/MEETH Financial Statements 12-2025.xlsx',
    2026: 'L:/2026 Pilar Plant Files/Monthly Expense Reports/9. September/MEETH/MEETH Financial Statements 09-2026 v2.xlsx',  
}  
SHEET = 'OP Summary'  
DEPT_KEYWORD = 'Chemo'          # matches 'Private Amb - Chemotherapy'

SECTIONS = ['Volume', 'Revenue']  
MONTHS = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',  
          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']      # no YTD  
METRICS = ['Actual', 'Budget', 'Variance']


# ---------- Helpers ----------  
def strip_cell(x):  
    return x.strip() if isinstance(x, str) else x


def to_num(col):  
    s = (col.astype(str)  
            .str.strip()  
            .str.replace(',', '', regex=False)  
            .str.replace(r'^\((.*)\)$', r'-\1', regex=True))  
    return pd.to_numeric(s, errors='coerce')


def parse_block(block):  
    block = block.reset_index(drop=True)

    hdr = block.index[block.eq('Actual').any(axis=1)][0]  
    metric_row = block.iloc[hdr]  
    dept_row = block.iloc[hdr - 1]  
    entity_row = block.iloc[hdr - 2]

    group = (metric_row == 'Actual').cumsum()  
    is_metric = metric_row.isin(METRICS)

    dept = dept_row.groupby(group).transform('first')  
    entity = entity_row.groupby(group).transform('first')

    labels = block.iloc[:, 0].astype(str).str.strip()  
    rows = labels.isin(MONTHS)  
    metric_cols = is_metric[is_metric].index

    data = block.loc[rows, metric_cols].apply(to_num)  
    data.index = pd.Index(labels[rows].values, name='month')  
    data.columns = pd.MultiIndex.from_arrays(  
        [entity[metric_cols], dept[metric_cols], metric_row[metric_cols]],  
        names=['entity', 'department', 'metric'],  
    )  
    return data


def chemo_only(df):  
    mask = df.columns.get_level_values('department').str.contains(  
        DEPT_KEYWORD, case=False, na=False  
    )  
    return df.loc[:, mask]


def load_chemo(path, year):  
    raw = pd.read_excel(path, sheet_name=SHEET, header=None)  
    raw = raw.apply(lambda col: col.map(strip_cell))

    col0 = raw.iloc[:, 0].astype(str)  
    starts = [col0[col0 == s].index[0] for s in SECTIONS]  
    ends = starts[1:] + [len(raw)]

    tables = {name: parse_block(raw.iloc[s:e])  
              for name, s, e in zip(SECTIONS, starts, ends)}

    out = (  
        pd.concat(  
            {name: chemo_only(df).unstack() for name, df in tables.items()},  
            names=['section'],  
        )  
        .rename('value')  
        .reset_index()  
        .dropna(subset=['value'])  
    )  
    out.insert(0, 'year', year)  
    return out


# ---------- Load all files and combine ----------  
chemo_long = pd.concat(  
    [load_chemo(path, year) for year, path in FILES.items()],  
    ignore_index=True,  
)

chemo_long['month'] = pd.to_datetime(  
    chemo_long['year'].astype(str) + '-' + chemo_long['month'].astype(str) + '-01',  
    format='%Y-%b-%d',)

chemo_long = (chemo_long  
              .sort_values(['year', 'section', 'metric', 'month'])  
              .reset_index(drop=True))

In [11]:
display(chemo_long)

chemo_long.to_excel('C:/Users/kbixby/OneDrive - Northwell Health/Documents/Financials/MEETH Infusion/chemo.xlsx')

,year,section,entity,department,metric,month,value
0,2025,Rate,MEETH,Private Amb - Chemotherapy,Actual,2025-01-01,6521.715099
1,2025,Rate,MEETH,Private Amb - Chemotherapy,Actual,2025-02-01,6305.836559
2,2025,Rate,MEETH,Private Amb - Chemotherapy,Actual,2025-03-01,6271.748895
3,2025,Rate,MEETH,Private Amb - Chemotherapy,Actual,2025-04-01,6313.757542
4,2025,Rate,MEETH,Private Amb - Chemotherapy,Actual,2025-05-01,6077.245556
...,...,...,...,...,...,...,...
184,2026,Volume,MEETH,Private Amb - Chemotherapy,Variance,2026-05-01,47.244764
185,2026,Volume,MEETH,Private Amb - Chemotherapy,Variance,2026-06-01,74.969240
186,2026,Volume,MEETH,Private Amb - Chemotherapy,Variance,2026-07-01,-1.030760
187,2026,Volume,MEETH,Private Amb - Chemotherapy,Variance,2026-08-01,7.607002


In [2]:
# ---------------- function to read protected excel ----------------

def read_protected_excel(file_path, password, sheet_name, skiprows):  
    """Read a single sheet from an Excel file that may or may not be password-protected."""  
    with open(file_path, 'rb') as f:  
        raw = io.BytesIO(f.read())

    workbook = raw  
    encrypted = False

    try:  
        raw.seek(0)  
        office_file = msoffcrypto.OfficeFile(raw)  
        if office_file.is_encrypted():  
            encrypted = True  
            decrypted = io.BytesIO()  
            office_file.load_key(password=password)  
            office_file.decrypt(decrypted)  
            workbook = decrypted  
    except FileFormatError:  
        workbook = raw  
    except InvalidKeyError:  
        raise RuntimeError(f"Wrong password for {os.path.basename(file_path)}")

    print(f"    ({'encrypted' if encrypted else 'not encrypted'})")

    workbook.seek(0)  
    df = pd.read_excel(workbook, sheet_name=sheet_name, skiprows=skiprows)  
    return df

# ---------------- config ----------------

load_dotenv()  
password = os.getenv("EPIC_INFUSION_PW")

epic_path = 'L:/2026 Pilar Plant Files/Requests from Departments/EPIC/Transactions (HB)_2026-11-12T15_26_00.xlsx'
sheet_name = 'Data'
skiprows = 10

try:  
    epic_infusion = read_protected_excel(epic_path, password, sheet_name, skiprows)  
except Exception as e:  
    print(f"  ERROR: {e}")

print(f"\n--- Summary ---")  
print(f"Epic Data: {epic_infusion.shape}")  


    (encrypted)

--- Summary ---
Epic Data: (13884, 17)


In [4]:
infusion_cases = (
    epic_infusion[['Start Date', 'Slices by Procedure CPT Code Detail.1', 'Hospital Account ID']]
    .drop_duplicates()
    .rename(columns={'Slices by Procedure CPT Code Detail.1': 'CPT Code'}))

display(infusion_cases)

,Start Date,CPT Code,Hospital Account ID
0,2026-01-01,96365 - HC IV INFUSION THERAPY/PROPHYLAXIS /DX...,1055000023836
1,2026-01-01,96365 - HC IV INFUSION THERAPY/PROPHYLAXIS /DX...,1055000023835
2,2026-01-01,96365 - HC IV INFUSION THERAPY/PROPHYLAXIS /DX...,1055000023837
3,2026-01-01,96365 - HC IV INFUSION THERAPY/PROPHYLAXIS /DX...,1055000020536
4,2026-01-01,96365 - HC IV INFUSION THERAPY/PROPHYLAXIS /DX...,1055000022105
...,...,...,...
13879,2026-10-01,99195 - HC PHLEBOTOMY THERAPEUTIC SEPARATE PRO...,1055000073636
13880,2026-10-01,99195 - HC PHLEBOTOMY THERAPEUTIC SEPARATE PRO...,1055000070425
13881,2026-10-01,99195 - HC PHLEBOTOMY THERAPEUTIC SEPARATE PRO...,1055000063413
13882,2026-10-01,99195 - HC PHLEBOTOMY THERAPEUTIC SEPARATE PRO...,1055000060967


In [7]:
infusion_types = pd.DataFrame(  
    [       ('96401 - HC CHEMOTX ADMN SUBQ/IM NON-HORMONAL ANTI-NEO', 'Infusion Chemotherapy'),
                ('96402 - HC CHEMOTX ADMN SUBQ/IM HORMONAL ANTI-NEO', 'Infusion Chemotherapy'),
               ('96409 - HC CHEMOTX ADMN IV PUSH TQ 1/1ST SBST/DRUG', 'Infusion Chemotherapy'),
                  ('96411 - HC CHEMOTX ADMN IV PUSH TQ EA SBST/DRUG', 'Infusion Chemotherapy'),
        ('96413 - HC CHEMOTX ADMN IV NFS TQ UP 1 HR 1/1ST SBST/DRUG', 'Infusion Chemotherapy'),
                  ('36415 - HC COLLECTION VENOUS BLOOD VENIPUNCTURE', 'Labs'),
                    ('36430 - HC TRANSFUSION BLOOD/BLOOD COMPONENTS', 'Injection/Transfusion'),
           ('96360 - HC IV INFUSION HYDRATION INITIAL 31 MIN-1 HOUR', 'Infusion Non-Chemotherapy'),
       ('96365 - HC IV INFUSION THERAPY/PROPHYLAXIS /DX 1ST TO 1 HR', 'Infusion Non-Chemotherapy'),
         ('96372 - HC THERAPEUTIC PROPHYLACTIC/DX INJECTION SUBQ/IM', 'Injection/Transfusion'),
        ('96374 - HC THER PROPH/DX NJX IV PUSH SINGLE/1ST SBST/DRUG', 'Injection/Transfusion'),
        ('96377 - HC APPL ON-BODY INJECTOR FOR TIMED SUBQ INJECTION', 'Injection/Transfusion'),
       ('96523 - HC IRRIGAJ IMPLNTD VENOUS ACCESS DRUG DELIVERY SYS', 'Injection/Transfusion'),
             ('99195 - HC PHLEBOTOMY THERAPEUTIC SEPARATE PROCEDURE', 'Phlebotomy')

           ],  
    columns=['CPT Code', 'Category']
)

inf_cat = pd.merge(infusion_cases, infusion_types, how='left', on='CPT Code')

inf_group = inf_cat.groupby(['Start Date', 'Category'])['Hospital Account ID'].count().reset_index()
inf_month = inf_cat.groupby(['Start Date'])['Hospital Account ID'].count().reset_index()
display(inf_group)
display(inf_month)

,Start Date,Category,Hospital Account ID
0,2026-01-01,Infusion Chemotherapy,419
1,2026-01-01,Infusion Non-Chemotherapy,275
2,2026-01-01,Injection/Transfusion,226
3,2026-01-01,Labs,272
4,2026-01-01,Phlebotomy,13
5,2026-02-01,Infusion Chemotherapy,418
6,2026-02-01,Infusion Non-Chemotherapy,233
7,2026-02-01,Injection/Transfusion,257
8,2026-02-01,Labs,219
9,2026-02-01,Phlebotomy,21


,Start Date,Hospital Account ID
0,2026-01-01,1207
1,2026-02-01,1149
2,2026-03-01,1418
3,2026-04-01,1420
4,2026-05-01,1394
5,2026-06-01,1524
6,2026-07-01,1445
7,2026-08-01,1376
8,2026-09-01,1433
9,2026-10-01,1481
